<a href="https://colab.research.google.com/github/Prempalepu/Task1_Big-data-analysis/blob/main/Task1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count

# Create Spark session
spark = SparkSession.builder \
    .appName("Big Data Analysis") \
    .getOrCreate()

# Load dataset with proper CSV handling (FIXED)
df = spark.read.csv(
    "/content/Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    quote='"',       # handles quotes in data
    escape='"',      # handles escaped quotes
    multiLine=True  # handles multi-line values
)

# Check columns
print("Columns:", df.columns)

# Convert Sales column to numeric (FIXED)
df = df.withColumn("Sales", col("Sales").cast("double"))

# Remove invalid rows where Sales is null
df = df.filter(col("Sales").isNotNull())

# Show schema after fix
print("Schema after cleaning:")
df.printSchema()

# Show sample data
df.show(5)

# -------------------------------
# ANALYSIS
# -------------------------------

# 1. Total Sales
total_sales = df.select(sum("Sales")).collect()[0][0]
print(f"Total Sales: {total_sales}")

# 2. Average Sales
avg_sales = df.select(avg("Sales")).collect()[0][0]
print(f"Average Sales: {avg_sales}")

# 3. Sales by Category
category_sales = df.groupBy("Category") \
    .agg(sum("Sales").alias("Total_Sales"))

print("Sales by Category:")
category_sales.show()

# 4. Top Selling Products
top_products = df.groupBy("Product Name") \
    .agg(sum("Sales").alias("Total_Sales")) \
    .orderBy(col("Total_Sales").desc())

print("Top Selling Products:")
top_products.show(5)

# 5. Orders per Region
orders_region = df.groupBy("Region") \
    .agg(count("Order ID").alias("Total_Orders"))

print("Orders per Region:")
orders_region.show()

# Stop Spark session
spark.stop()

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Schema after cleaning:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sale